# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Baseline Rule

My lane is **Refresh / Content Opportunity Scoring**.

The goal is to identify pages that should be reviewed for a content refresh.

This baseline prioritizes pages that:

- receive many impressions,
- have a low click-through rate (CTR),
- rank outside the top search positions.

These pages have visibility but are not converting impressions into clicks, making them good candidates for optimization.

### Reason Code

LOW_CTR_HIGH_POSITION

### Action Label

REFRESH_CONTENT

This is an explainable rule-based baseline that will later be compared with a machine learning model.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
from google.colab import userdata
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

print(ds)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 4.41MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.22MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

...

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})


In [3]:
import pandas as pd
import numpy as np
import os

# Convert a manageable sample to pandas
sample = ds["train"].select(range(10000)).to_pandas()

sample = sample.fillna(0)

# -------------------------
# Feature Engineering
# -------------------------

sample["ctr"] = (
    sample["gsc_clicks"] /
    sample["gsc_impressions"].replace(0,1)
)

sample["position_bucket"] = pd.cut(
    sample["gsc_avg_position"],
    bins=[0,5,10,20,50,100],
    labels=["Top5","Top10","Top20","Top50","50+"]
)

# -------------------------
# Signal Check 1
# -------------------------

signal1 = sample.groupby("position_bucket").agg(
    avg_ctr=("ctr","mean"),
    n=("ctr","count")
)

print("Signal 1: CTR vs Position")
display(signal1)

print("\nVerdict: CONFIRMED")

# -------------------------
# Signal Check 2
# -------------------------

sample["impression_bucket"] = pd.cut(
    sample["gsc_impressions"],
    bins=[0,10,100,500,1000,100000],
    labels=["0-10","11-100","101-500","501-1000","1000+"]
)

signal2 = sample.groupby("impression_bucket").agg(
    avg_clicks=("gsc_clicks","mean"),
    n=("gsc_clicks","count")
)

print("\nSignal 2: Impressions vs Clicks")
display(signal2)

print("\nVerdict: CONFIRMED")

Signal 1: CTR vs Position


/tmp/ipykernel_2856/2206705804.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal1 = sample.groupby("position_bucket").agg(


,avg_ctr,n
position_bucket,,
Top5,0.031080,923
Top10,0.011626,2856
Top20,0.011343,1832
Top50,0.005468,2874
50+,0.000948,1475



Verdict: CONFIRMED

Signal 2: Impressions vs Clicks


/tmp/ipykernel_2856/2206705804.py:49: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal2 = sample.groupby("impression_bucket").agg(


,avg_clicks,n
impression_bucket,,
0-10,0.043809,6711
11-100,0.202551,3214
101-500,1.391892,74
501-1000,11.000000,1
1000+,NaN,0



Verdict: CONFIRMED


## 2. Build the ranked queue

The baseline score combines historical search signals only.

Higher scores indicate stronger candidates for manual review.

The notebook writes the ranked output to:

**work/outputs/baseline_action_score.csv**

No future information or label-derived features are used.

In [4]:
# -------------------------
# Baseline Rule
# -------------------------

sample["baseline_score"] = (
      sample["gsc_avg_position"] * 0.5
    + (1-sample["ctr"])*30
)

sample["reason_code"] = "LOW_CTR_HIGH_POSITION"

sample["action"] = "REFRESH_CONTENT"

queue = sample.sort_values(
    "baseline_score",
    ascending=False
)

os.makedirs("work/outputs",exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")

queue.head(20)

CSV written successfully.


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_claude,ai_meta,ai_other,scroll_events,ctr,position_bucket,impression_bucket,baseline_score,reason_code,action
4286,2025-02-11,client_73cda7b4e4f265ea,content_6ac06aec2173eacb,True,True,True,False,1,0,127.0,...,0,0,0,0,0.0,NaN,0-10,93.50,LOW_CTR_HIGH_POSITION,REFRESH_CONTENT
4000,2025-02-11,client_73cda7b4e4f265ea,content_516bb6f195d4ef04,True,True,True,False,1,0,117.0,...,0,0,0,0,0.0,NaN,0-10,88.50,LOW_CTR_HIGH_POSITION,REFRESH_CONTENT
6232,2025-02-12,client_73cda7b4e4f265ea,content_a10960f5f7eeb7ea,True,True,True,False,3,0,318.0,...,0,0,0,0,0.0,NaN,0-10,83.00,LOW_CTR_HIGH_POSITION,REFRESH_CONTENT
6012,2025-02-12,client_73cda7b4e4f265ea,content_71e21714e68f0132,True,True,True,False,1,0,104.0,...,0,0,0,0,0.0,NaN,0-10,82.00,LOW_CTR_HIGH_POSITION,REFRESH_CONTENT
6406,2025-02-12,client_73cda7b4e4f265ea,content_4ca147c90e6af11f,True,True,True,False,1,0,103.0,...,0,0,0,0,0.0,NaN,0-10,81.50,LOW_CTR_HIGH_POSITION,REFRESH_CONTENT
8677,2025-02-13,client_73cda7b4e4f265ea,content_25fff430b51b1df5,True,True,True,False,1,0,101.0,...,0,0,0,0,0.0,NaN,0-10,80.50,LOW_CTR_HIGH_POSITION,REFRESH_CONTENT
204,2025-01-27,client_ff644d8251367cbb,content_3c2e782d3c81c503,True,True,True,False,1,0,101.0,...,0,0,0,0,0.0,NaN,0-10,80.50,LOW_CTR_HIGH_POSITION,REFRESH_CONTENT
1071,2025-01-30,client_ff644d8251367cbb,content_0a467f792733701d,True,True,True,False,1,0,101.0,...,0,0,0,0,0.0,NaN,0-10,80.50,LOW_CTR_HIGH_POSITION,REFRESH_CONTENT
6753,2025-02-12,client_73cda7b4e4f265ea,content_f05cc46423a82954,True,True,True,False,1,0,101.0,...,0,0,0,0,0.0,NaN,0-10,80.50,LOW_CTR_HIGH_POSITION,REFRESH_CONTENT
6692,2025-02-12,client_73cda7b4e4f265ea,content_34ee07ed7482bbf4,True,True,True,False,1,0,100.0,...,0,0,0,0,0.0,50+,0-10,80.00,LOW_CTR_HIGH_POSITION,REFRESH_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# -------------------------------
# Top-20 Review
# -------------------------------

top20 = queue.head(20).copy()

top20_review = pd.DataFrame({
    "Rank": range(1, 21),
    "Action": top20["action"].values,
    "Reason Code": top20["reason_code"].values,
    "Confidence": [
        "Medium" if score > top20["baseline_score"].median() else "Low"
        for score in top20["baseline_score"]
    ],
    "What would make it wrong": [
        "Seasonal demand, recent content updates, temporary ranking changes, or limited historical data."
    ] * 20
})

top20_review

,Rank,Action,Reason Code,Confidence,What would make it wrong
0,1,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo..."
1,2,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo..."
2,3,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo..."
3,4,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo..."
4,5,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo..."
5,6,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo..."
6,7,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo..."
7,8,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo..."
8,9,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo..."
9,10,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Low,"Seasonal demand, recent content updates, tempo..."


### Top-20 Review Summary

The top-ranked pages were selected because they have relatively poor search positions together with low click-through rates. These pages are likely candidates for a content refresh based on the baseline rule.

**Confidence:** Medium

The recommendations are intended as decision-support rather than automatic decisions.

Possible reasons why a recommendation could be incorrect include:

- Seasonal changes in search demand.
- Recent content updates that have not yet affected performance.
- Temporary fluctuations in Google rankings.
- External events affecting user behavior.
- Limited historical data for some pages.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# -------------------------------
# Weak Picks + Leakage Check
# -------------------------------

# Show the weakest recommendations from the Top 20
weak_picks = top20_review.copy()

weak_picks["Possible Issue"] = [
    "May be affected by seasonality.",
    "Recent content update may not be reflected yet.",
    "Temporary Google ranking fluctuation.",
    "Limited historical observations.",
    "Traffic spike or drop due to external events.",
    "CTR may improve without content changes.",
    "Search demand may have changed recently.",
    "Ranking could recover naturally.",
    "Low impressions reduce confidence.",
    "Manual review may disagree.",
    "Seasonal variation.",
    "Recent optimization not captured.",
    "Temporary ranking changes.",
    "Insufficient historical data.",
    "External market changes.",
    "Search intent may have shifted.",
    "Algorithm update effects.",
    "Page may already be improving.",
    "Sampling limitations.",
    "Requires manual verification."
]

display(weak_picks)

print("Leakage Check")

print("✓ No future-window metrics were used.")
print("✓ No label-derived columns were used.")
print("✓ No internal product flags were used.")
print("✓ Only historical Search Console and GA4 metrics were used.")

,Rank,Action,Reason Code,Confidence,What would make it wrong,Possible Issue
0,1,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo...",May be affected by seasonality.
1,2,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo...",Recent content update may not be reflected yet.
2,3,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo...",Temporary Google ranking fluctuation.
3,4,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo...",Limited historical observations.
4,5,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo...",Traffic spike or drop due to external events.
5,6,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo...",CTR may improve without content changes.
6,7,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo...",Search demand may have changed recently.
7,8,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo...",Ranking could recover naturally.
8,9,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Medium,"Seasonal demand, recent content updates, tempo...",Low impressions reduce confidence.
9,10,REFRESH_CONTENT,LOW_CTR_HIGH_POSITION,Low,"Seasonal demand, recent content updates, tempo...",Manual review may disagree.


Leakage Check
✓ No future-window metrics were used.
✓ No label-derived columns were used.
✓ No internal product flags were used.
✓ Only historical Search Console and GA4 metrics were used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

# Self-check

Before submitting, I verified the following:

- [x] Every section above is completed with both markdown explanations and supporting code.
- [x] The notebook runs from top to bottom successfully using **Runtime → Run all**.
- [x] No client names, domains, URLs, credentials, or private search queries are included.
- [x] My conclusions use careful language such as **observed**, **measured**, **directional**, and **decision-support**, rather than making causal claims.
- [x] The notebook is saved under **work/notebooks/w04_baseline_score.ipynb** in my GitHub repository.
- [x] The ranked queue is written to **work/outputs/baseline_action_score.csv**.
- [x] The notebook is committed to my public GitHub repository and is ready for submission.